# Notebook de EDA + Limpeza de dados Produtos

## Configuração de Ambiente


### Bibliotecas python

In [423]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

### Caminho base do projeto

In [424]:
BASE_PATH = Path().resolve()

while BASE_PATH.name != "lh-nautical-data-project":
    BASE_PATH = BASE_PATH.parent

print(f"BASE PATH: {BASE_PATH}")

BASE PATH: /Users/richardgomes/lh-nautical-data-project


### Caminhos para os dados raw e staging

In [425]:
DATA_PATH = BASE_PATH / "data"
RAW_PATH = DATA_PATH / "raw"
STAGING_PATH = DATA_PATH / "staging"

In [426]:
print(f"RAW PATH: {RAW_PATH}")
print(f"STAGING PATH: {STAGING_PATH}")

RAW PATH: /Users/richardgomes/lh-nautical-data-project/data/raw
STAGING PATH: /Users/richardgomes/lh-nautical-data-project/data/staging


##### Leitura e carregamento dos dados de produtos (produtos_raw.csv)

In [427]:
df_produtos = pd.read_csv(RAW_PATH / "produtos_raw.csv", encoding="utf-8")

In [428]:
df_produtos.shape

(157, 4)

In [429]:
df_produtos.head(10)

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz
5,Transponder AIS Vector,R$ 11820.21,6,Eletrunicos
6,Radar AIS Zen,R$ 19518.77,7,eLeTrÔnIcOs
7,GPS AIS Zen,R$ 4984.15,8,E L E T R Ô N I C O S
8,Transponder AIS Titan Pulse,R$ 39705.5,9,Eletronicoz
9,Piloto Automático Simrad Titan Flux Magnum,R$ 32033.04,10,eletrônicos


In [430]:
df_produtos.info()

<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   name             157 non-null    str  
 1   price            157 non-null    str  
 2   code             157 non-null    int64
 3   actual_category  157 non-null    str  
dtypes: int64(1), str(3)
memory usage: 5.0 KB


##### Tratamento da coluna price

In [431]:
df_produtos["price"] = (
    df_produtos["price"]
    .str.replace("R$", "", regex=False)
    .str.strip()
    .astype(float)
)

In [432]:
print(df_produtos['price'].dtype)

float64


In [433]:
df_produtos.info()

<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             157 non-null    str    
 1   price            157 non-null    float64
 2   code             157 non-null    int64  
 3   actual_category  157 non-null    str    
dtypes: float64(1), int64(1), str(2)
memory usage: 5.0 KB


#### Tratamento da coluna actual_category (categorias)

In [434]:
df_produtos["actual_category"].unique()

<StringArray>
[          'ELETRONICOS', 'E L E T R Ô N I C O S',           'Eletrunicos',
           'Eletronicoz',           'eLeTrÔnIcOs',           'eletrônicos',
           'Eletrônicos',          'Eletroniscos',           'Eletronicos',
           'eletronicos',           'EletrônicoS',           'ELEtRÔNICOS',
             'PROPULSAO',             'Propulção',                  'Prop',
            'Propulssão',             'propulsao',     'P R O P U L S Ã O',
              'Propução',             'propulsão',             'pRoPuLsÃo',
             'Propulçao',             'Propulsam',             'PrOpUlSãO',
             'Ancoragem',             'AnCoRaGeM',             'Encoragem',
            'Ancoraguem',              'Ancorajm',             'AncorageM',
     'A N C O R A G E M',             'ANCORAGEM',             'aNcOrAgEm',
             'Ancorajem',              'Encoragi',             'ancoragem',
             'Ancorajen',             'AncorajeM',             'Ancoragen'

In [435]:
df_produtos["actual_category_clean"] = (
    df_produtos["actual_category"]
    .str.lower()
    .str.strip()
)

df_produtos["actual_category_clean"].unique()

<StringArray>
[          'eletronicos', 'e l e t r ô n i c o s',           'eletrunicos',
           'eletronicoz',           'eletrônicos',          'eletroniscos',
             'propulsao',             'propulção',                  'prop',
            'propulssão',     'p r o p u l s ã o',              'propução',
             'propulsão',             'propulçao',             'propulsam',
             'ancoragem',             'encoragem',            'ancoraguem',
              'ancorajm',     'a n c o r a g e m',             'ancorajem',
              'encoragi',             'ancorajen',             'ancoragen']
Length: 24, dtype: str

Remoção de acentos da coluna actual_category

In [436]:
import unicodedata

def remover_acentos(texto):
    if isinstance(texto, str):
        return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto

df_produtos["actual_category_clean"] = df_produtos["actual_category_clean"].apply(remover_acentos)

df_produtos["actual_category_clean"].unique()

<StringArray>
[          'eletronicos', 'e l e t r o n i c o s',           'eletrunicos',
           'eletronicoz',          'eletroniscos',             'propulsao',
             'propulcao',                  'prop',            'propulssao',
     'p r o p u l s a o',              'propucao',             'propulsam',
             'ancoragem',             'encoragem',            'ancoraguem',
              'ancorajm',     'a n c o r a g e m',             'ancorajem',
              'encoragi',             'ancorajen',             'ancoragen']
Length: 21, dtype: str

Remoção de espaços internos nas categorias

In [437]:
df_produtos["actual_category_clean"] = (
    df_produtos["actual_category_clean"]
    .str.replace(" ", "", regex=False)
)



In [438]:
df_produtos["actual_category_clean"].unique()

<StringArray>
[ 'eletronicos',  'eletrunicos',  'eletronicoz', 'eletroniscos',
    'propulsao',    'propulcao',         'prop',   'propulssao',
     'propucao',    'propulsam',    'ancoragem',    'encoragem',
   'ancoraguem',     'ancorajm',    'ancorajem',     'encoragi',
    'ancorajen',    'ancoragen']
Length: 18, dtype: str

Padronização das categorias

In [439]:
def padronizar_categoria(cat):
    if "eletro" in cat:
        return "eletronicos"
    elif "prop" in cat:
        return "propulsao"
    elif "ancor" in cat:
        return "ancoragem"
    else:
        return "outros"

In [440]:
df_produtos["category_final"] = df_produtos["actual_category_clean"].apply(padronizar_categoria)

df_produtos["category_final"].unique()

<StringArray>
['eletronicos', 'outros', 'propulsao', 'ancoragem']
Length: 4, dtype: str

In [441]:
df_produtos[df_produtos["category_final"] == "outros"]["actual_category_clean"].unique()

<StringArray>
['eletrunicos', 'encoragem', 'encoragi']
Length: 3, dtype: str

In [442]:
def padronizar_categoria(cat):
    if "eletro" in cat or "eletru" in cat:
        return "eletronicos"
    elif "prop" in cat:
        return "propulsao"
    elif "ancor" in cat or "encor" in cat:
        return "ancoragem"
    else:
        return "outros"

In [443]:
df_produtos["category_final"] = df_produtos["actual_category_clean"].apply(padronizar_categoria)

df_produtos["category_final"].unique()

<StringArray>
['eletronicos', 'propulsao', 'ancoragem']
Length: 3, dtype: str

In [444]:
df_produtos.head()

,name,price,code,actual_category,actual_category_clean,category_final
0,Transponder AIS Maré Magnum,33122.52,1,ELETRONICOS,eletronicos,eletronicos
1,Transponder Furuno Marlin,13998.15,2,ELETRONICOS,eletronicos,eletronicos
2,Radar Furuno Pulse Leviathan,9024.19,3,E L E T R Ô N I C O S,eletronicos,eletronicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,Eletrunicos,eletrunicos,eletronicos
4,Piloto Automático Furuno Storm,23669.01,5,Eletronicoz,eletronicoz,eletronicos


In [445]:
def padronizar_categoria(cat):
    if not isinstance(cat, str):
        return "outros"
    
    cat = cat.lower()
    cat = cat.replace(" ", "")  # resolve "e l e t r o n i c o s"
    
    if "eletr" in cat:
        return "eletronicos"
    elif "prop" in cat:
        return "propulsao"
    elif "ancor" in cat or "encor" in cat:
        return "ancoragem"
    else:
        return "outros"

In [446]:
df_produtos["category_final"] = df_produtos["actual_category_clean"].apply(padronizar_categoria)

In [447]:
df_produtos.head()

,name,price,code,actual_category,actual_category_clean,category_final
0,Transponder AIS Maré Magnum,33122.52,1,ELETRONICOS,eletronicos,eletronicos
1,Transponder Furuno Marlin,13998.15,2,ELETRONICOS,eletronicos,eletronicos
2,Radar Furuno Pulse Leviathan,9024.19,3,E L E T R Ô N I C O S,eletronicos,eletronicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,Eletrunicos,eletrunicos,eletronicos
4,Piloto Automático Furuno Storm,23669.01,5,Eletronicoz,eletronicoz,eletronicos


In [448]:
df_produtos = df_produtos.drop(columns=["actual_category", "actual_category_clean"])

Ajuste final das colunas (padronização para português)

In [449]:
df_produtos = df_produtos.rename(columns={
    "codigo_produto": "id_produto",
    "preco": "preco_base",
    "categoria": "categoria"
})

df_produtos

,name,price,code,category_final
0,Transponder AIS Maré Magnum,33122.52,1,eletronicos
1,Transponder Furuno Marlin,13998.15,2,eletronicos
2,Radar Furuno Pulse Leviathan,9024.19,3,eletronicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,eletronicos
4,Piloto Automático Furuno Storm,23669.01,5,eletronicos
...,...,...,...,...
152,Corrente Delta Vox Ion,495.98,146,ancoragem
153,Corrente Danforth Force Leviathan Impulse,3030.08,147,ancoragem
154,Âncora Delta Force Barracuda Mako,4785.56,148,ancoragem
155,Cabo de Nylon Bruce Core,1163.62,149,ancoragem


In [450]:
df_produtos = df_produtos.rename(columns={
    "code": "id_produto",
    "price": "preco_base_produto",
    "category_final": "categoria_produto",
    "name": "nome_produto"
})

Padronizando nomes das colunas para o Power BI

In [451]:
df_produtos

,nome_produto,preco_base_produto,id_produto,categoria_produto
0,Transponder AIS Maré Magnum,33122.52,1,eletronicos
1,Transponder Furuno Marlin,13998.15,2,eletronicos
2,Radar Furuno Pulse Leviathan,9024.19,3,eletronicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,eletronicos
4,Piloto Automático Furuno Storm,23669.01,5,eletronicos
...,...,...,...,...
152,Corrente Delta Vox Ion,495.98,146,ancoragem
153,Corrente Danforth Force Leviathan Impulse,3030.08,147,ancoragem
154,Âncora Delta Force Barracuda Mako,4785.56,148,ancoragem
155,Cabo de Nylon Bruce Core,1163.62,149,ancoragem


In [452]:
df_produtos.info()

<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nome_produto        157 non-null    str    
 1   preco_base_produto  157 non-null    float64
 2   id_produto          157 non-null    int64  
 3   categoria_produto   157 non-null    str    
dtypes: float64(1), int64(1), str(2)
memory usage: 5.0 KB


In [453]:
df_produtos.shape

(157, 4)

In [454]:
df_produtos.to_csv(STAGING_PATH / "stg_produtos.csv", index=False)